# 03b — Student Transformer Training (FIXED V2)

**Bug fixes applied:**
- Bug 1: Noam LR schedule — optimizer base_lr=1.0, LambdaLR returns absolute LR
- Bug 2: Dynamic warmup steps (~5% of total, bounded [200,2000])
- Bug 3: Early stopping guarded by MIN_EPOCHS_BEFORE_STOP
- Bug 4: RESUME_TRAINING=False — explicit opt-in only; new run name avoids old checkpoints
- Bug 5: Checkpoint saves scaler_state, model_cfg, full train_cfg, tokenizer hash, rng_states
- Bug 6: Loss via nn.CrossEntropyLoss(ignore_index=PAD, label_smoothing=0.1)
- Bug 7: Data validation + sample decoding before training
- Bug 8: Overfit sanity check on 64-example subset
- Bug 9: Quick-eval uses seeded deterministic shuffled subset, not first-N
- Bug 10: Atomic checkpoint save (temp file + os.replace)

**Run name:** `student_beam_M1_optA_fixed_v2`

**Platform:** Kaggle T4 GPU | PyTorch 2.x | Python 3.12

In [1]:
!pip install -q torch sentencepiece tqdm pandas sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.2 MB/s eta 0:00:00


In [2]:
import hashlib, json, math, os, random, shutil, tempfile, time
import numpy as np, pandas as pd
from pathlib import Path
from typing import List, Dict, Tuple, Optional
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from torch.amp import autocast, GradScaler
from tqdm.auto import tqdm
import sentencepiece as spm
import sacrebleu

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0),
          '| Memory: {:.1f} GB'.format(
              torch.cuda.get_device_properties(0).total_memory / 1e9))

Device: cuda
GPU: Tesla T4 | Memory: 15.6 GB


In [3]:
from pathlib import Path

print("Available input folders:")
for path in Path("/kaggle/input").iterdir():
    print(path)

print("\nAvailable cache files:")
cache_files = list(Path("/kaggle/input").rglob("*.pt"))

for path in cache_files:
    print(path)

Available input folders:
/kaggle/input/datasets

Available cache files:
/kaggle/input/datasets/nemooo1205/mhd-checkpoint/student_beam_M10_merged_optA_fixed_v2_latest.pt
/kaggle/input/datasets/nemooo1205/mhd-checkpoint/student_beam_M10_merged_optA_fixed_v2_best_val.pt
/kaggle/input/datasets/nemooo1205/mhd-data-final/notebooks/notebooks/models/cache/cache_top_p_M10.pt
/kaggle/input/datasets/nemooo1205/mhd-data-final/notebooks/notebooks/models/cache/cache_beam_M10.pt
/kaggle/input/datasets/nemooo1205/mhd-data-final/notebooks/notebooks/models/cache/cache_flores_dev.pt
/kaggle/input/datasets/nemooo1205/mhd-data-final/notebooks/notebooks/models/cache/cache_dbs_M10.pt
/kaggle/input/datasets/nemooo1205/mhd-data-final/notebooks/notebooks/models/cache/cache_flores_devtest.pt
/kaggle/input/datasets/nemooo1205/mhd-data-final/notebooks/notebooks/models/cache/cache_top_k_M10.pt
/kaggle/input/datasets/nemooo1205/mhd-data-final/notebooks/notebooks/models/cache/cache_beam_M1.pt
/kaggle/input/datasets/n

In [4]:
from pathlib import Path

cache_candidates = list(
    Path("/kaggle/input").rglob("cache_beam_M10_merged.pt")
)

if not cache_candidates:
    raise FileNotFoundError(
        "cache_beam_M10_merged.pt was not found anywhere inside /kaggle/input"
    )

CACHE_FILE = cache_candidates[0]
CACHE_DIR = CACHE_FILE.parent

print("Cache file:", CACHE_FILE)
print("Cache directory:", CACHE_DIR)

cache_data = torch.load(
    CACHE_FILE,
    map_location="cpu",
    weights_only=False
)

Cache file: /kaggle/input/datasets/nemooo1205/mhd-data-final/notebooks/notebooks/models/cache/cache_beam_M10_merged.pt
Cache directory: /kaggle/input/datasets/nemooo1205/mhd-data-final/notebooks/notebooks/models/cache


In [5]:
# ── PATH DISCOVERY ──────────────────────────────────────────────────────
# Dynamically locate vocab_info.json and SentencePiece model.
# On Kaggle: search /kaggle/input recursively, rank by relevance.
# Locally: search relative to cwd.
# NEVER hardcodes any dataset slug.

IS_KAGGLE   = Path('/kaggle/working').exists()
WORK_ROOT   = Path('/kaggle/working') if IS_KAGGLE else Path.cwd()
INPUT_ROOT  = Path('/kaggle/input')   if IS_KAGGLE else Path.cwd()

MODEL_DIR   = WORK_ROOT / 'models' / 'student_run'
RESULTS_DIR = WORK_ROOT / 'results'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Locate vocab_info.json candidates ─────────────────────────────────
vocab_candidates = list(INPUT_ROOT.rglob('vocab_info.json'))
# Also check local paths
_local_candidates = [
    Path.cwd() / 'notebooks' / 'models' / 'vocab_info.json',
    Path.cwd() / 'notebooks' / 'notebooks' / 'models' / 'vocab_info.json',
    Path.cwd().parent / 'notebooks' / 'models' / 'vocab_info.json',
]
for _lc in _local_candidates:
    if _lc.exists() and _lc not in vocab_candidates:
        vocab_candidates.append(_lc)

print('vocab_info.json candidates found ({:d}):'.format(len(vocab_candidates)))
for _c in vocab_candidates:
    print('  ', _c)

if not vocab_candidates:
    raise FileNotFoundError('No vocab_info.json found. Run 03a first.')

# ── Rank candidates by: has spm model nearby, has cache nearby, then depth ─
def _score_candidate(path):
    d = path.parent
    score = 0
    if list(d.glob('*.model')): score += 10
    if list(d.rglob('cache_beam_M1.pt')): score += 8
    if list(d.rglob('cache_*.pt')): score += 4
    if list(d.glob('*.vocab')): score += 2
    # prefer shallower path (fewer parts = closer to root of dataset)
    score -= len(path.parts)
    return score

_ranked = sorted(vocab_candidates, key=_score_candidate, reverse=True)
print('Ranked candidates (best first):')
for _r in _ranked:
    print('  score={:d}  {}'.format(_score_candidate(_r), _r))

VOCAB_INFO_SRC = _ranked[0]
SRC_MODEL_DIR  = VOCAB_INFO_SRC.parent
print('Selected source directory:', SRC_MODEL_DIR)

# ── Copy required files (not whole directory) ─────────────────────────
_files_to_copy = ['vocab_info.json', 'shared_spm.model', 'shared_spm.vocab']
for _fname in _files_to_copy:
    _src_f = SRC_MODEL_DIR / _fname
    _dst_f = MODEL_DIR / _fname
    if _src_f.exists() and not _dst_f.exists():
        shutil.copy2(_src_f, _dst_f)
        print('Copied', _fname)
    elif _dst_f.exists():
        print('Already present:', _fname)
    else:
        print('WARNING: not found:', _src_f)

# ── Load vocab_info ───────────────────────────────────────────────────
VOCAB_INFO_PATH = MODEL_DIR / 'vocab_info.json'
if not VOCAB_INFO_PATH.exists():
    # fallback: use source directly
    VOCAB_INFO_PATH = VOCAB_INFO_SRC

with open(VOCAB_INFO_PATH, 'r', encoding='utf-8') as _f:
    vi = json.load(_f)

VOCAB_SIZE  = int(vi['vocab_size'])
PAD_ID      = int(vi['pad_id'])
BOS_ID      = int(vi['bos_id'])
EOS_ID      = int(vi['eos_id'])
MAX_LENGTH  = int(vi['max_length'])

# ── Locate SentencePiece model ────────────────────────────────────────
_spm_candidates = list(MODEL_DIR.glob('*.model'))
if not _spm_candidates:
    _spm_candidates = list(SRC_MODEL_DIR.glob('*.model'))
if not _spm_candidates:
    _spm_candidates = list(INPUT_ROOT.rglob('shared_spm.model')) if IS_KAGGLE else []
    _spm_candidates += list(Path.cwd().rglob('shared_spm.model'))

if not _spm_candidates:
    raise FileNotFoundError('No *.model SentencePiece file found.')

# Copy best spm model to MODEL_DIR if needed
SPM_MODEL_PATH = MODEL_DIR / 'shared_spm.model'
if not SPM_MODEL_PATH.exists():
    shutil.copy2(_spm_candidates[0], SPM_MODEL_PATH)
    print('Copied spm model from', _spm_candidates[0])

sp = spm.SentencePieceProcessor(model_file=str(SPM_MODEL_PATH))


print(f"Cache file: {CACHE_FILE}")
print(f"Cache directory: {CACHE_DIR}")


# ── Validate tokenizer ────────────────────────────────────────────────
assert sp.get_piece_size() == VOCAB_SIZE, \
    'Tokenizer vocab size {:d} != expected {:d}'.format(sp.get_piece_size(), VOCAB_SIZE)
assert sp.pad_id()  == PAD_ID or PAD_ID == 0,  'PAD mismatch'
assert sp.bos_id()  == BOS_ID, 'BOS mismatch: sp={} cfg={}'.format(sp.bos_id(), BOS_ID)
assert sp.eos_id()  == EOS_ID, 'EOS mismatch: sp={} cfg={}'.format(sp.eos_id(), EOS_ID)

print()
print('vocab_size =', VOCAB_SIZE, '| PAD={} BOS={} EOS={}'.format(PAD_ID, BOS_ID, EOS_ID))
print('MAX_LENGTH =', MAX_LENGTH)
print('SPM model  =', SPM_MODEL_PATH)
print('Cache dir  =', CACHE_DIR)
print('Tokenizer validated OK')

vocab_info.json candidates found (8):
   /kaggle/input/datasets/nemooo1205/mhd-data-final/notebooks/notebooks/models/vocab_info.json
   /kaggle/input/datasets/nemooo1205/mhd-data-final/notebooks/notebooks/models/top_k_M10/vocab_info.json
   /kaggle/input/datasets/nemooo1205/mhd-data-final/notebooks/notebooks/models/top_p_M10/vocab_info.json
   /kaggle/input/datasets/nemooo1205/mhd-data-final/notebooks/notebooks/models/beam_M10/vocab_info.json
   /kaggle/input/datasets/nemooo1205/mhd-data-final/notebooks/notebooks/models/shared/vocab_info.json
   /kaggle/input/datasets/nemooo1205/mhd-data-final/notebooks/notebooks/models/dbs_M10/vocab_info.json
   /kaggle/input/datasets/nemooo1205/mhd-data-final/notebooks/notebooks/models/mbr_M10/vocab_info.json
   /kaggle/input/datasets/nemooo1205/mhd-data-final/notebooks/notebooks/models/beam_M1/vocab_info.json
Ranked candidates (best first):
  score=14  /kaggle/input/datasets/nemooo1205/mhd-data-final/notebooks/notebooks/models/vocab_info.json
  scor

In [6]:
# ── USER SETTINGS ───────────────────────────────────────────────────────
DATASET         = 'beam_M10_merged'   # beam_M1 | beam_M10 | beam_M10_merged | top_p_M10 | top_k_M10 | dbs_M10 | mbr_M10
MODEL_SIZE      = 'A'         # A or B

# Training hyperparameters
BATCH_SIZE      = 32
GRADIENT_ACCUM  = 2
MAX_EPOCHS      = 60
LABEL_SMOOTHING = 0.1
WEIGHT_DECAY    = 1e-4
CLIP_GRAD       = 1.0
USE_FP16        = torch.cuda.is_available()
NUM_WORKERS     = 2

# Early stopping -- only active after MIN_EPOCHS_BEFORE_STOP
EARLY_STOP_PATIENCE     = 8
MIN_EPOCHS_BEFORE_STOP  = 10

# Safe resume -- must be explicitly enabled (Bug 4 fix)
RESUME_TRAINING   = True
RESUME_CHECKPOINT = "/kaggle/input/datasets/nemooo1205/mhd-checkpoint/student_beam_M10_merged_optA_fixed_v2_best_val.pt" 
# set to a .pt path string to resume from specific file

# Overfit sanity test -- disable only after successful first run (Bug 8 fix)
RUN_OVERFIT_TEST       = True
OVERFIT_EXAMPLES       = 64
OVERFIT_STEPS          = 400
OVERFIT_LOSS_THRESHOLD = 0.5

# Quick-eval: seeded deterministic subset (Bug 9 fix)
QUICK_EVAL_N    = 200
QUICK_EVAL_SEED = 42

# Model architectures
MODEL_CFGS = {
    'A': dict(d_model=512, nhead=8, num_encoder_layers=6, num_decoder_layers=6, d_ff=2048, dropout=0.3),
    'B': dict(d_model=128, nhead=4, num_encoder_layers=2, num_decoder_layers=2, d_ff=512,  dropout=0.1),
}
model_cfg = MODEL_CFGS[MODEL_SIZE]

# New run name -- never conflicts with old broken runs (Bug 4 fix)
RUN_NAME    = 'student_{}_opt{}_fixed_v2'.format(DATASET, MODEL_SIZE)
CKPT_DIR    = MODEL_DIR / RUN_NAME
CKPT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_BEST_VAL  = CKPT_DIR / '{}_best_val.pt'.format(RUN_NAME)
CKPT_BEST_CHRF = CKPT_DIR / '{}_best_chrf.pt'.format(RUN_NAME)
CKPT_LATEST    = CKPT_DIR / '{}_latest.pt'.format(RUN_NAME)
LOG_CSV        = RESULTS_DIR / 'training_log_{}.csv'.format(RUN_NAME)

print('Run name   :', RUN_NAME)
print('Checkpoint :', CKPT_DIR)
print('Log CSV    :', LOG_CSV)
print('FP16       :', USE_FP16)
print('Model cfg  :', model_cfg)

Run name   : student_beam_M10_merged_optA_fixed_v2
Checkpoint : /kaggle/working/models/student_run/student_beam_M10_merged_optA_fixed_v2
Log CSV    : /kaggle/working/results/training_log_student_beam_M10_merged_optA_fixed_v2.csv
FP16       : True
Model cfg  : {'d_model': 512, 'nhead': 8, 'num_encoder_layers': 6, 'num_decoder_layers': 6, 'd_ff': 2048, 'dropout': 0.3}


In [7]:
# ── DATA LOADING AND VALIDATION ────────────────────────────────────────
# Bug 7 fix: full validation + sample decoding before training

class CachedDataset(Dataset):
    def __init__(self, data: List[Dict]):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

def collate_fn(batch):
    src_padded = pad_sequence([ex['src'] for ex in batch], batch_first=True, padding_value=PAD_ID)
    tgt_padded = pad_sequence([ex['tgt'] for ex in batch], batch_first=True, padding_value=PAD_ID)
    return {'src': src_padded, 'tgt': tgt_padded}

# ── Load cache ────────────────────────────────────────────────────────
cache_path = CACHE_DIR / 'cache_{}.pt'.format(DATASET)
assert cache_path.exists(), 'Cache not found: {}. Run 03a first.'.format(cache_path)
print('Loading cache:', cache_path)
raw_data = torch.load(str(cache_path), weights_only=False)
print('  {:d} pairs loaded'.format(len(raw_data)))

# ── Length statistics ─────────────────────────────────────────────────
def _len_stats(tensors, name):
    lengths = [len(t) for t in tensors]
    arr = np.array(lengths)
    print('  {} len: min={} mean={:.1f} max={} p95={}'.format(
        name, arr.min(), arr.mean(), arr.max(), int(np.percentile(arr, 95))))
    return arr

src_tensors = [ex['src'] for ex in raw_data]
tgt_tensors = [ex['tgt'] for ex in raw_data]
_src_arr = _len_stats(src_tensors, 'src')
_tgt_arr = _len_stats(tgt_tensors, 'tgt')

# Count empty targets (len <= 2 means only BOS+EOS)
_empty_tgt = sum(1 for t in tgt_tensors if len(t) <= 2)
print('  Empty targets: {:d} ({:.1f}%)'.format(_empty_tgt, 100 * _empty_tgt / len(raw_data)))

# Duplicate check on a sample
_src_strs  = [tuple(ex['src'].tolist()) for ex in raw_data[:2000]]
_dup_count = len(_src_strs) - len(set(_src_strs))
print('  Duplicate src (first 2k sample): {:d} ({:.1f}%)'.format(
    _dup_count, 100 * _dup_count / len(_src_strs)))

# ── Token ID validation on 500-example sample ─────────────────────────
_rng = random.Random(99)
_sample_idx = _rng.sample(range(len(raw_data)), min(500, len(raw_data)))
_bad_ids = 0
for _i in _sample_idx:
    for _tid in raw_data[_i]['src'].tolist() + raw_data[_i]['tgt'].tolist():
        if not (0 <= _tid < VOCAB_SIZE):
            _bad_ids += 1
assert _bad_ids == 0, 'Found {:d} token IDs outside [0, {:d})'.format(_bad_ids, VOCAB_SIZE)
print('  Token ID validation: OK (500-example sample)')

# ── Decode 10 random pairs ────────────────────────────────────────────
print()
print('--- 10 random decoded pairs ---')
_show_idx = _rng.sample(range(len(raw_data)), 10)
for _ii, _si in enumerate(_show_idx):
    _ex   = raw_data[_si]
    _stok = _ex['src'].tolist()
    _ttok = _ex['tgt'].tolist()
    # strip BOS/EOS for display
    _stok = [t for t in _stok if t not in (BOS_ID, EOS_ID, PAD_ID)]
    _ttok = [t for t in _ttok if t not in (BOS_ID, EOS_ID, PAD_ID)]
    _src_str = sp.decode(_stok)
    _tgt_str = sp.decode(_ttok)
    print('  [{:d}] SRC: {}'.format(_ii, _src_str))
    print('       TGT: {}'.format(_tgt_str))

# ── Sanity checks ────────────────────────────────────────────────────
_abort_reasons = []
if _empty_tgt / len(raw_data) > 0.20:
    _abort_reasons.append('>20% empty targets')
# Check first 200 for identity (src==tgt) and ASCII in src
_identity_count = 0
_no_ascii_count = 0
for _ex in raw_data[:200]:
    _st = [t for t in _ex['src'].tolist() if t not in (BOS_ID, EOS_ID, PAD_ID)]
    _tt = [t for t in _ex['tgt'].tolist() if t not in (BOS_ID, EOS_ID, PAD_ID)]
    if _st == _tt: _identity_count += 1
    _sdec = sp.decode(_st)
    if not any(ord(c) < 128 for c in _sdec): _no_ascii_count += 1
if _identity_count / 200 > 0.20:
    _abort_reasons.append('>20% src==tgt (identity translations)')
if _no_ascii_count / 200 > 0.50:
    _abort_reasons.append('>50% source sentences contain no ASCII (not English?)')
if _abort_reasons:
    raise RuntimeError('Data validation FAILED: ' + '; '.join(_abort_reasons))
print()
print('Data validation passed.')

# ── Train/val split ──────────────────────────────────────────────────
val_size   = max(200, int(0.05 * len(raw_data)))
train_size = len(raw_data) - val_size
_gen = torch.Generator().manual_seed(SEED)
train_ds, val_ds = random_split(CachedDataset(raw_data), [train_size, val_size], generator=_gen)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)

print('Train: {:d} | Val: {:d}'.format(len(train_ds), len(val_ds)))
print('Train batches/epoch:', len(train_loader))

# ── FLORES dev data ──────────────────────────────────────────────────
# Bug fix: also search the raw flores json in additional locations beyond CACHE_DIR.
# If 03a was run with a different working directory or dataset layout, raw_flores_dev.json
# may not be in CACHE_DIR. Search broadly so FLORES eval is never silently skipped.
_flores_raw_path = CACHE_DIR / 'raw_flores_dev.json'
if not _flores_raw_path.exists():
    # Secondary search: look anywhere reachable
    _flores_search_roots = [INPUT_ROOT] if IS_KAGGLE else [Path.cwd()]
    for _root in _flores_search_roots:
        for _found in _root.rglob('raw_flores_dev.json'):
            _flores_raw_path = _found
            print('Found raw_flores_dev.json at:', _found)
            break
        else:
            continue
        break
if _flores_raw_path.exists():
    with open(str(_flores_raw_path), 'r', encoding='utf-8') as _ff:
        _flores_raw = json.load(_ff)
    FLORES_DEV_SRC = _flores_raw['src']
    FLORES_DEV_REF = _flores_raw['ref']
    print('FLORES dev: {:d} sentences'.format(len(FLORES_DEV_SRC)))
else:
    FLORES_DEV_SRC = []
    FLORES_DEV_REF = []
    print('WARNING: raw_flores_dev.json not found in CACHE_DIR or any subdirectory.')
    print('  Searched:', CACHE_DIR)
    print('  Quick-eval BLEU/chrF++ will report 0 every epoch until this is fixed.')
    print('  To fix: re-run 03a_tokenizer_and_data.ipynb and include the cache folder in your Kaggle dataset.')
    # Hard assertion so the user cannot miss this
    raise RuntimeError(
        'raw_flores_dev.json not found. FLORES quick-eval is required. '
        'Re-run 03a and upload the cache folder (including raw_flores_dev.json) to your Kaggle dataset.'
    )

Loading cache: /kaggle/input/datasets/nemooo1205/mhd-data-final/notebooks/notebooks/models/cache/cache_beam_M10_merged.pt
  1000000 pairs loaded
  src len: min=12 mean=40.0 max=128 p95=72
  tgt len: min=6 mean=34.8 max=128 p95=65
  Empty targets: 0 (0.0%)
  Duplicate src (first 2k sample): 1800 (90.0%)
  Token ID validation: OK (500-example sample)

--- 10 random decoded pairs ---
  [0] SRC: A national coalition working to get lesbian, gay, bisexual and transgendered people health coverage under the Affordable Care Act has been...
       TGT: Ushirikiano wa kitaifa unaofanya kazi kupata chanjo ya afya kwa watu wa jinsia moja, wa jinsia moja, bisexual na transgender chini ya Sheria ya Affordable Care imekuwa...
  [1] SRC: I'm writing a letter complaining of this nonsense to the FDA... Oh wait! that's right they don't reply to any of their mail (despite the staff increase) and probably don't even read it unless its return address is some lawfirm. I guess its up to the people who own the 

In [8]:
# ── MODEL DEFINITION ────────────────────────────────────────────────────

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 512):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.max_len = max_len
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.size(1)
        assert seq_len <= self.max_len, \
            'Sequence length {:d} exceeds PositionalEncoding max_len {:d}'.format(
                seq_len, self.max_len)
        return self.dropout(x + self.pe[:, :seq_len])


class StudentTransformer(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, nhead: int,
                 num_encoder_layers: int, num_decoder_layers: int,
                 d_ff: int, dropout: float, max_len: int = 512, pad_id: int = 0):
        super().__init__()
        self.d_model = d_model
        self.pad_id  = pad_id
        self.embedding   = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        nn.init.normal_(self.embedding.weight, mean=0.0, std=d_model ** -0.5)
        with torch.no_grad():
            self.embedding.weight[pad_id].zero_()
        
        self.pos_enc     = PositionalEncoding(d_model, dropout, max_len)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=d_ff, dropout=dropout, batch_first=True,
        )
        # output projection with weight tying -- share embedding weights
        self.output_proj = nn.Linear(d_model, vocab_size, bias=False)
        self.output_proj.weight = self.embedding.weight  # weight tying
        _tied_id = id(self.output_proj.weight)
        for p in self.parameters():
            if p.dim() > 1 and id(p) != _tied_id:
                nn.init.xavier_uniform_(p)

    def pad_mask(self, ids: torch.Tensor) -> torch.Tensor:
        """Return boolean mask: True where ids == PAD (padding positions).""",
        return ids == self.pad_id

    def forward(self, src: torch.Tensor, tgt: torch.Tensor) -> torch.Tensor:
        scale    = self.d_model ** 0.5
        tgt_len  = tgt.size(1)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_len, device=src.device, dtype=torch.bool)
        src_emb  = self.pos_enc(self.embedding(src) * scale)
        tgt_emb  = self.pos_enc(self.embedding(tgt) * scale)
        out = self.transformer(
            src_emb, tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=self.pad_mask(src),
            tgt_key_padding_mask=self.pad_mask(tgt),
            memory_key_padding_mask=self.pad_mask(src),
        )
        return self.output_proj(out)


def build_model(cfg: dict = None) -> StudentTransformer:
    cfg = cfg or model_cfg
    return StudentTransformer(
        vocab_size=VOCAB_SIZE,
        max_len=MAX_LENGTH + 64,
        pad_id=PAD_ID,
        **cfg,
    ).to(device)


model = build_model()
n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Model Option {}: {:.2f}M total params | {:.2f}M trainable'.format(
    MODEL_SIZE, n_params / 1e6, n_trainable / 1e6))

Model Option A: 60.52M total params | 60.52M trainable


In [9]:
# ── LR SCHEDULE VERIFICATION AND OPTIMIZER SETUP ───────────────────────
# Bug 1 fix: optimizer base_lr=1.0 so LambdaLR output IS the absolute LR.
# Bug 2 fix: warmup_steps computed dynamically from actual dataset size.

steps_per_epoch    = math.ceil(len(train_loader) / GRADIENT_ACCUM)
total_opt_steps    = steps_per_epoch * MAX_EPOCHS
# ~5% of total optimizer steps, bounded [200, 2000]
warmup_steps       = int(max(200, min(2000, 0.05 * total_opt_steps)))
warmup_epochs_approx = warmup_steps / max(steps_per_epoch, 1)

print('steps_per_epoch    :', steps_per_epoch)
print('total_opt_steps    :', total_opt_steps)
print('warmup_steps       :', warmup_steps)
print('warmup_epochs_approx: {:.1f}'.format(warmup_epochs_approx))


def get_scheduler(optimizer, d_model: int, warmup_steps: int):
    """
    Noam (Transformer) learning rate schedule.

    With optimizer base_lr=1.0, LambdaLR returns the absolute LR directly.
    Formula: lr(step) = d_model^(-0.5) * min(step^(-0.5), step * warmup^(-1.5))

    Peak LR occurs at step=warmup_steps:
      peak = d_model^(-0.5) * warmup_steps^(-0.5)
    For d_model=512, warmup=596: peak approx 1.8e-3
    """
    def lr_lambda(step: int) -> float:
        step = max(step, 1)
        return (d_model ** -0.5) * min(step ** -0.5, step * warmup_steps ** -1.5)
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# Bug 1 fix: base_lr = 1.0 so lr_lambda IS the effective LR
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=LABEL_SMOOTHING).to(device)
optimizer = torch.optim.Adam(
    model.parameters(), lr=1.0,
    betas=(0.9, 0.98), eps=1e-9, weight_decay=WEIGHT_DECAY,
)
scheduler = get_scheduler(optimizer, model_cfg['d_model'], warmup_steps)
scaler    = GradScaler(enabled=USE_FP16)

# ── LR diagnostic ────────────────────────────────────────────────────
def _noam_lr(step, d_model, w_steps):
    step = max(step, 1)
    return (d_model ** -0.5) * min(step ** -0.5, step * w_steps ** -1.5)

_d  = model_cfg['d_model']
_ws = warmup_steps
_probe_steps = [
    1, 10, 100,
    max(1, _ws // 2), _ws, _ws * 2,
    max(1, total_opt_steps // 2), max(1, total_opt_steps)
]
print()
print('--- LR Diagnostic (Noam schedule) ---')
print('{:<20s}  {:>12s}'.format('step', 'lr'))
print('-' * 36)
_probe_lrs = []
for _s in _probe_steps:
    _lr = _noam_lr(_s, _d, _ws)
    _probe_lrs.append(_lr)
    _tag = ''
    if _s == _ws: _tag = '  <-- PEAK'
    print('{:<20d}  {:>12.6e}{}'.format(_s, _lr, _tag))

peak_lr = _noam_lr(_ws, _d, _ws)
print()
print('Peak LR = {:.6e}  (at warmup step {:d})'.format(peak_lr, _ws))

# ── Assertions ───────────────────────────────────────────────────────
for _lr in _probe_lrs:
    assert math.isfinite(_lr) and _lr > 0, 'LR not finite/positive: {}'.format(_lr)
assert 1e-5 <= peak_lr <= 1e-1, \
    'Peak LR {:.2e} out of expected range [1e-5, 1e-1]'.format(peak_lr)
assert _noam_lr(1, _d, _ws) < peak_lr, 'LR at step 1 should be < peak (warmup must increase)'
print('All LR assertions passed.')
print('Criterion, Optimizer, Scheduler, Scaler ready.')

steps_per_epoch    : 14844
total_opt_steps    : 890640
warmup_steps       : 2000
warmup_epochs_approx: 0.1

--- LR Diagnostic (Noam schedule) ---
step                            lr
------------------------------------
1                     4.941059e-07
10                    4.941059e-06
100                   4.941059e-05
1000                  4.941059e-04
2000                  9.882118e-04  <-- PEAK
4000                  6.987712e-04
445320                6.622606e-05
890640                4.682890e-05

Peak LR = 9.882118e-04  (at warmup step 2000)
All LR assertions passed.
Criterion, Optimizer, Scheduler, Scaler ready.


In [10]:
# ── OVERFIT SANITY TEST ──────────────────────────────────────────────────
# Bug 8 fix: verify the pipeline can overfit a tiny subset before full training.
# Uses a SEPARATE fresh model -- main model is untouched.

if RUN_OVERFIT_TEST:
    print('Running overfit sanity test on {:d} examples for {:d} steps...'.format(
        OVERFIT_EXAMPLES, OVERFIT_STEPS))
    print('This uses a separate fresh model and will be discarded after the test.')

    # Build a fresh model (do NOT use the main model)
    _ov_cfg = dict(model_cfg)
    _ov_cfg['dropout'] = 0.0   # disable dropout for the overfit sanity check only
    _ov_model = build_model(_ov_cfg)
    _ov_crit  = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=0.0).to(device)
    _ov_optim = torch.optim.Adam(_ov_model.parameters(), lr=3e-4, betas=(0.9, 0.98), eps=1e-9)
    _ov_warmup_steps = 30
    
    def _ov_lr_lambda(step):
        step = max(step, 1)
        return min(1.0, step / _ov_warmup_steps)

    _ov_sched = torch.optim.lr_scheduler.LambdaLR(_ov_optim, _ov_lr_lambda)

    # Tiny subset -- same OVERFIT_EXAMPLES every step
    _ov_indices = list(range(min(OVERFIT_EXAMPLES, len(train_ds))))
    from torch.utils.data import Subset
    _ov_subset  = Subset(train_ds, _ov_indices)
    _ov_loader  = DataLoader(_ov_subset, batch_size=min(16, OVERFIT_EXAMPLES),
                             shuffle=True, collate_fn=collate_fn)

    _ov_model.train()
    _ov_step  = 0
    _ov_loss  = float('inf')
    _ov_cycle = iter(_ov_loader)

    for _ov_step in range(1, OVERFIT_STEPS + 1):
        try:
            _batch = next(_ov_cycle)
        except StopIteration:
            _ov_cycle = iter(_ov_loader)
            _batch    = next(_ov_cycle)

        _src = _batch['src'].to(device)
        _tgt = _batch['tgt'].to(device)
        _tgt_in  = _tgt[:, :-1]
        _tgt_out = _tgt[:, 1:]

        _ov_optim.zero_grad()
        _logits = _ov_model(_src, _tgt_in)
        _B, _T, _V = _logits.shape
        _ov_loss_val = _ov_crit(_logits.reshape(_B * _T, _V), _tgt_out.reshape(_B * _T))
        _ov_loss_val.backward()
        torch.nn.utils.clip_grad_norm_(_ov_model.parameters(), 1.0)
        _ov_optim.step()
        _ov_sched.step()
        _ov_loss = _ov_loss_val.item()

        if _ov_step % 20 == 0 or _ov_step == 1:
            print('  overfit step {:>4d}/{:d}  loss={:.4f}'.format(
                _ov_step, OVERFIT_STEPS, _ov_loss))

    print('Final overfit loss: {:.4f} (threshold: {:.2f})'.format(
        _ov_loss, OVERFIT_LOSS_THRESHOLD))

    # Decode 3 training examples with greedy
    print()
    print('--- Overfit greedy decode (3 training examples) ---')
    _ov_model.eval()
    for _oi in range(min(3, len(_ov_indices))):
        _ex   = train_ds[_ov_indices[_oi]]
        _stok = [t for t in _ex['src'].tolist() if t not in (PAD_ID,)]
        _ttok = [t for t in _ex['tgt'].tolist() if t not in (BOS_ID, EOS_ID, PAD_ID)]
        _src_str = sp.decode([t for t in _stok if t not in (BOS_ID, EOS_ID)])
        _ref_str = sp.decode(_ttok)
        # greedy decode using overfit model
        with torch.no_grad():
            _s_t = torch.tensor([_stok], dtype=torch.long, device=device)
            _scale = _ov_model.d_model ** 0.5
            _s_emb = _ov_model.pos_enc(_ov_model.embedding(_s_t) * _scale)
            _s_pad = _ov_model.pad_mask(_s_t)
            _mem   = _ov_model.transformer.encoder(_s_emb, src_key_padding_mask=_s_pad)
            _dec   = [BOS_ID]
            for _ in range(80):
                _tg  = torch.tensor([_dec], dtype=torch.long, device=device)
                _tm  = nn.Transformer.generate_square_subsequent_mask(len(_dec), device=device)
                _te  = _ov_model.pos_enc(_ov_model.embedding(_tg) * _scale)
                _out = _ov_model.transformer.decoder(_te, _mem, tgt_mask=_tm,
                                                     memory_key_padding_mask=_s_pad)
                _nid = _ov_model.output_proj(_out[0, -1]).argmax().item()
                if _nid == EOS_ID: break
                _dec.append(_nid)
            _hyp = sp.decode(_dec[1:])
        print('  SRC:', _src_str)
        print('  REF:', _ref_str)
        print('  HYP:', _hyp)
        print()

    if _ov_loss >= OVERFIT_LOSS_THRESHOLD:
        raise RuntimeError(
            'Overfit test FAILED: loss={:.4f} >= threshold={:.2f}. '
            'Pipeline is broken. Fix before full training.'.format(
                _ov_loss, OVERFIT_LOSS_THRESHOLD))

    print('Overfit sanity test PASSED (loss={:.4f} < {:.2f})'.format(
        _ov_loss, OVERFIT_LOSS_THRESHOLD))

    # Discard test model and free GPU memory
    del _ov_model, _ov_crit, _ov_optim, _ov_loader, _ov_subset
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print('Test model deleted. GPU memory freed.')
else:
    print('Overfit sanity test SKIPPED (RUN_OVERFIT_TEST=False)')

Running overfit sanity test on 64 examples for 400 steps...
This uses a separate fresh model and will be discarded after the test.
  overfit step    1/400  loss=10.8699
  overfit step   20/400  loss=8.4098
  overfit step   40/400  loss=5.9291
  overfit step   60/400  loss=3.9338
  overfit step   80/400  loss=1.8412
  overfit step  100/400  loss=0.3586
  overfit step  120/400  loss=0.1330
  overfit step  140/400  loss=0.1056
  overfit step  160/400  loss=0.0197
  overfit step  180/400  loss=0.0095
  overfit step  200/400  loss=0.0156
  overfit step  220/400  loss=0.0140
  overfit step  240/400  loss=0.0169
  overfit step  260/400  loss=0.0211
  overfit step  280/400  loss=0.0152
  overfit step  300/400  loss=0.0148
  overfit step  320/400  loss=0.0090
  overfit step  340/400  loss=0.0089
  overfit step  360/400  loss=0.0087
  overfit step  380/400  loss=0.0206
  overfit step  400/400  loss=0.0587
Final overfit loss: 0.0587 (threshold: 0.50)

--- Overfit greedy decode (3 training example

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


  SRC: Currently running on Android 2.3 the software update for Samsung Galaxy Ace 2 to Android 4.0 is planned to be released in the near future.
  REF: Hivi sasa kuendesha kwenye Android 2.3 sasisho la programu ya Samsung Galaxy Ace 2 kwa Android 4.0 imepangwa kutolewa hivi karibuni.
  HYP: Hivi sasa kuendesha kwenye Android 2.3 sasisho la programu ya Samsung Galaxy Ace 2 kwa Android 4.0 imepangwa kutolewa hivi karibuni.

  SRC: Linda was now letting loose with a whole string of obscenities, although avoiding motherfucker, and Tommy was looking pretty annoyed. When he reached his limit and felt he had heard enough, he reached for the winch’s remote switch, which was hanging there.
  REF: Sasa Linda alikuwa akijiweka huru na mfululizo wa mambo machafu, ingawa alikuwa akijiepusha na mwana wazimu, na Tommy alikuwa akionekana kuwa amekasirika sana.Alipokuwa amefikia kikomo chake na kuhisi kwamba alikuwa amesikia vya kutosha, alifikia kibadilishaji cha mbali cha winch, ambacho kilikuwa kin

In [11]:
# ── RESUME HANDLING ─────────────────────────────────────────────────────
# Bug 4 fix: only resume when RESUME_TRAINING=True; verify checkpoint fields.

START_EPOCH   = 1
GLOBAL_STEP   = 0
BEST_VAL_LOSS = float('inf')
BEST_DEV_CHRF = -1.0
NO_IMPROVE    = 0
training_log  = []

if RESUME_TRAINING:
    # Find checkpoint path
    if RESUME_CHECKPOINT is not None:
        _ckpt_path = Path(RESUME_CHECKPOINT)
    elif CKPT_LATEST.exists():
        _ckpt_path = CKPT_LATEST
    else:
        # Search for any checkpoint with this run name
        _found = list(CKPT_DIR.glob('*.pt'))
        if not _found:
            raise FileNotFoundError(
                'RESUME_TRAINING=True but no checkpoint found in {}'.format(CKPT_DIR))
        _ckpt_path = sorted(_found, key=lambda p: p.stat().st_mtime)[-1]
    print('Resuming from:', _ckpt_path)
    _ckpt = torch.load(str(_ckpt_path), map_location=device, weights_only=False)

    # Verify checkpoint fields
    _required_ckpt_keys = [
        'model_cfg', 'vocab_size', 'dataset', 'run_name',
        'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict',
        'epoch', 'global_step'
    ]
    for _k in _required_ckpt_keys:
        assert _k in _ckpt, 'Checkpoint missing key: {}'.format(_k)

    # Verify compatibility
    assert _ckpt['model_cfg'] == model_cfg, \
        'model_cfg mismatch: ckpt={} current={}'.format(_ckpt['model_cfg'], model_cfg)
    assert _ckpt['vocab_size'] == VOCAB_SIZE, \
        'vocab_size mismatch: ckpt={} current={}'.format(_ckpt['vocab_size'], VOCAB_SIZE)
    assert _ckpt['dataset'] == DATASET, \
        'dataset mismatch: ckpt={} current={}'.format(_ckpt['dataset'], DATASET)

    # Load states
    model.load_state_dict(_ckpt['model_state_dict'])
    optimizer.load_state_dict(_ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(_ckpt['scheduler_state_dict'])
    if 'scaler_state_dict' in _ckpt and _ckpt['scaler_state_dict']:
        scaler.load_state_dict(_ckpt['scaler_state_dict'])

    # Move optimizer state to correct device
    for _os in optimizer.state.values():
        for _k, _v in _os.items():
            if isinstance(_v, torch.Tensor):
                _os[_k] = _v.to(device)

    START_EPOCH   = _ckpt['epoch'] + 1
    GLOBAL_STEP   = _ckpt.get('global_step', 0)
    BEST_VAL_LOSS = _ckpt.get('best_val_loss', float('inf'))
    BEST_DEV_CHRF = _ckpt.get('best_dev_chrf', -1.0)
    NO_IMPROVE    = _ckpt.get('no_improve', 0)

    if 'rng_states' in _ckpt:
        _rng_states = _ckpt['rng_states']
        random.setstate(_rng_states['python'])
        np.random.set_state(_rng_states['numpy'])

        torch.set_rng_state(_rng_states['cpu_torch'].cpu().to(torch.uint8))
        
        if torch.cuda.is_available() and 'cuda_torch' in _rng_states:
            torch.cuda.set_rng_state(_rng_states['cuda_torch'].cpu().to(torch.uint8))

    # Load existing log
    if LOG_CSV.exists():
        training_log = pd.read_csv(str(LOG_CSV)).to_dict('records')

    print('Resumed at epoch {} | best_val_loss={:.4f} | best_chrF++={:.2f}'.format(
        START_EPOCH, BEST_VAL_LOSS, BEST_DEV_CHRF))
else:
    print('Starting fresh run:', RUN_NAME)

Resuming from: /kaggle/input/datasets/nemooo1205/mhd-checkpoint/student_beam_M10_merged_optA_fixed_v2_best_val.pt
Resumed at epoch 2 | best_val_loss=3.0265 | best_chrF++=-1.00


In [12]:
# ── TRAINING HELPER FUNCTIONS ───────────────────────────────────────────

def train_epoch(model, loader, optimizer, scheduler, scaler, criterion,
                global_step: int, gradient_accum: int) -> Tuple[float, int]:
    """
    Run one training epoch with gradient accumulation and AMP.
    Returns (avg_loss_original_scale, new_global_step).
    """
    model.train()
    total_loss   = 0.0
    total_tokens = 0
    optimizer.zero_grad()

    for batch_idx, batch in enumerate(tqdm(loader, desc='Train', leave=False)):
        src     = batch['src'].to(device, non_blocking=True)
        tgt     = batch['tgt'].to(device, non_blocking=True)
        tgt_in  = tgt[:, :-1]
        tgt_out = tgt[:, 1:]

        with autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu',
                      enabled=USE_FP16):
            logits = model(src, tgt_in)                          # (B, T, V)
            B, T, V = logits.shape
            # Loss divided by gradient_accum for accumulation
            loss = criterion(logits.reshape(B * T, V), tgt_out.reshape(B * T))
            loss_accum = loss / gradient_accum

        scaler.scale(loss_accum).backward()

        # Count non-pad tokens for logging (use original loss scale)
        n_tokens = (tgt_out != PAD_ID).sum().item()
        total_loss   += loss.item() * n_tokens
        total_tokens += n_tokens

        # Detect NaN/Inf
        if not math.isfinite(loss.item()):
            print('WARNING: non-finite loss={:.4f} at batch {:d}'.format(
                loss.item(), batch_idx))

        # Optimizer step every gradient_accum batches
        if (batch_idx + 1) % gradient_accum == 0 or (batch_idx + 1) == len(loader):
            # Unscale before gradient clipping
            scaler.unscale_(optimizer)
            grad_norm = nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD)
            scaler.step(optimizer)
            scaler.update()
            # scheduler.step() ONLY after successful optimizer step
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

            if global_step % 50 == 0:
                _cur_lr = scheduler.get_last_lr()[0]
                print('  step={:>6d}  loss={:.4f}  lr={:.3e}  gnorm={:.3f}'.format(
                    global_step, loss.item(), _cur_lr, grad_norm))

    avg_loss = total_loss / max(total_tokens, 1)
    return avg_loss, global_step


@torch.no_grad()
def validate(model, loader, criterion) -> float:
    """Token-weighted average cross-entropy loss on validation set.""",
    model.eval()
    total_loss   = 0.0
    total_tokens = 0
    for batch in tqdm(loader, desc='Val', leave=False):
        src     = batch['src'].to(device, non_blocking=True)
        tgt     = batch['tgt'].to(device, non_blocking=True)
        tgt_in  = tgt[:, :-1]
        tgt_out = tgt[:, 1:]
        with autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu',
                      enabled=USE_FP16):
            logits = model(src, tgt_in)
            B, T, V = logits.shape
            loss = criterion(logits.reshape(B * T, V), tgt_out.reshape(B * T))
        n_tokens = (tgt_out != PAD_ID).sum().item()
        total_loss   += loss.item() * n_tokens
        total_tokens += n_tokens
    return total_loss / max(total_tokens, 1)


@torch.no_grad()
def greedy_translate(model, src_text: str, max_len: int = 128) -> str:
    """Greedy decode a single source string.""",
    model.eval()
    src_ids = ([BOS_ID]
               + sp.encode(src_text, add_bos=False, add_eos=False)[:MAX_LENGTH - 2]
               + [EOS_ID])
    src = torch.tensor([src_ids], dtype=torch.long, device=device)
    scale   = model.d_model ** 0.5
    src_emb = model.pos_enc(model.embedding(src) * scale)
    src_pad = model.pad_mask(src)
    memory  = model.transformer.encoder(src_emb, src_key_padding_mask=src_pad)
    decoded = [BOS_ID]
    for _ in range(max_len):
        tgt     = torch.tensor([decoded], dtype=torch.long, device=device)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(len(decoded), device=device, dtype=torch.bool)
        tgt_emb = model.pos_enc(model.embedding(tgt) * scale)
        out     = model.transformer.decoder(
            tgt_emb, memory, tgt_mask=tgt_mask,
            memory_key_padding_mask=src_pad,
        )
        next_id = model.output_proj(out[0, -1]).argmax().item()
        if next_id == EOS_ID:
            break
        decoded.append(next_id)
    return sp.decode(decoded[1:])   # strip BOS


# Bug 9 fix: build the eval subset ONCE with a seeded RNG
_eval_rng = random.Random(QUICK_EVAL_SEED)
if len(FLORES_DEV_SRC) > 0:
    _eval_indices  = list(range(len(FLORES_DEV_SRC)))
    _eval_rng.shuffle(_eval_indices)
    _eval_indices  = _eval_indices[:QUICK_EVAL_N]
    EVAL_SRC_FIXED = [FLORES_DEV_SRC[i] for i in _eval_indices]
    EVAL_REF_FIXED = [FLORES_DEV_REF[i] for i in _eval_indices]
    # 5 fixed examples to print every eval call
    EVAL_PRINT_IDX = _eval_indices[:5]
else:
    EVAL_SRC_FIXED = []
    EVAL_REF_FIXED = []
    EVAL_PRINT_IDX = []


def quick_flores_eval(model) -> Tuple[float, float, float, float, float]:
    """
    Greedy BLEU/chrF++ on the fixed seeded EVAL_SRC_FIXED subset.
    Returns (bleu, chrf, empty_pct, avg_len, unique_ratio).
    """
    if not EVAL_SRC_FIXED:
        return 0.0, 0.0, 0.0, 0.0, 0.0
    hyps = [greedy_translate(model, s)
            for s in tqdm(EVAL_SRC_FIXED, desc='QuickEval', leave=False)]
    bleu  = sacrebleu.corpus_bleu(hyps, [EVAL_REF_FIXED], tokenize='flores200').score
    chrf  = sacrebleu.corpus_chrf(hyps, [EVAL_REF_FIXED], word_order=2).score
    empty = sum(1 for h in hyps if len(h.strip()) == 0)
    empty_pct   = empty / len(hyps)
    lengths     = [len(h.split()) for h in hyps]
    avg_len     = float(np.mean(lengths)) if lengths else 0.0
    all_words   = [w for h in hyps for w in h.split()]
    unique_ratio = len(set(all_words)) / max(len(all_words), 1)
    # Print 5 fixed examples
    print('  --- 5 fixed quick-eval examples ---')
    for _pi in range(min(5, len(EVAL_SRC_FIXED))):
        print('    SRC:', EVAL_SRC_FIXED[_pi])
        print('    REF:', EVAL_REF_FIXED[_pi])
        print('    HYP:', hyps[_pi])
        print()
    return bleu, chrf, empty_pct, avg_len, unique_ratio


print('Training helpers ready.')
print('Fixed eval subset size:', len(EVAL_SRC_FIXED))

Training helpers ready.
Fixed eval subset size: 200


In [13]:
# ── CHECKPOINT SAVE/LOAD HELPERS ────────────────────────────────────────
# Bug 5 fix: save full set of required fields
# Bug 10 fix: atomic save via temp file + os.replace

# Compute tokenizer SHA-256 once
TOKENIZER_SHA256 = hashlib.sha256(
    Path(str(SPM_MODEL_PATH)).read_bytes()
).hexdigest()
print('Tokenizer SHA-256:', TOKENIZER_SHA256)

TRAIN_CFG = dict(
    batch_size=BATCH_SIZE,
    gradient_accum=GRADIENT_ACCUM,
    max_epochs=MAX_EPOCHS,
    label_smoothing=LABEL_SMOOTHING,
    weight_decay=WEIGHT_DECAY,
    clip_grad=CLIP_GRAD,
    use_fp16=USE_FP16,
    early_stop_patience=EARLY_STOP_PATIENCE,
    min_epochs_before_stop=MIN_EPOCHS_BEFORE_STOP,
    warmup_steps=warmup_steps,
    steps_per_epoch=steps_per_epoch,
    optimizer_base_lr=1.0,
    seed=SEED,
)

SPLIT_META = dict(
    total=len(raw_data),
    train=len(train_ds),
    val=len(val_ds),
    split_seed=SEED,
)

print('VOCAB_INFO:', vi)
print('TRAIN_CFG :', TRAIN_CFG)


def save_checkpoint(path, epoch: int, global_step: int,
                    model, optimizer, scheduler, scaler,
                    best_val_loss: float, best_dev_chrf: float,
                    no_improve: int) -> None:
    """
    Atomic checkpoint save.
    Writes to a temp file first, then os.replace() to final path.
    Bug 10 fix: prevents partial writes corrupting the checkpoint.
    Bug 5 fix: saves all required fields including scaler, model_cfg, etc.
    """
    _rng_states: Dict = {
        'python'    : random.getstate(),
        'numpy'     : np.random.get_state(),
        'cpu_torch' : torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        _rng_states['cuda_torch'] = torch.cuda.get_rng_state()

    payload = {
        'schema_version'       : 2,
        'epoch'                : epoch,
        'global_step'          : global_step,
        'model_state_dict'     : model.state_dict(),
        'optimizer_state_dict' : optimizer.state_dict(),
        'scheduler_state_dict' : scheduler.state_dict(),
        'scaler_state_dict'    : scaler.state_dict() if USE_FP16 else None,
        'best_val_loss'        : best_val_loss,
        'best_dev_chrf'        : best_dev_chrf,
        'no_improve'           : no_improve,
        'model_cfg'            : model_cfg,
        'run_name'             : RUN_NAME,
        'dataset'              : DATASET,
        'vocab_size'           : VOCAB_SIZE,
        'tokenizer_sha256'     : TOKENIZER_SHA256,
        'vocab_info'           : vi,
        'train_cfg'            : TRAIN_CFG,
        'split_meta'           : SPLIT_META,
        'rng_states'           : _rng_states,
    }

    _path = Path(path)
    _path.parent.mkdir(parents=True, exist_ok=True)
    # Write to temp file in same directory, then atomically replace
    _fd, _tmp = tempfile.mkstemp(dir=str(_path.parent), suffix='.pt.tmp')
    try:
        os.close(_fd)
        torch.save(payload, _tmp)
        os.replace(_tmp, str(_path))   # atomic on POSIX; near-atomic on Windows
    except Exception:
        try: os.unlink(_tmp)
        except OSError: pass
        raise

    print('  Saved: {}'.format(_path.name))


print('Checkpoint helpers ready.')

Tokenizer SHA-256: db7a654d238085cde75d9fcbd5fd422ab71653cf34123829f178dc162d99437b
VOCAB_INFO: {'vocab_size': 32000, 'pad_id': 0, 'unk_id': 1, 'bos_id': 2, 'eos_id': 3, 'max_length': 128, 'spm_model': 'c:\\Users\\nirmi\\Desktop\\MHD2\\notebooks\\models\\shared_spm.model'}
TRAIN_CFG : {'batch_size': 32, 'gradient_accum': 2, 'max_epochs': 60, 'label_smoothing': 0.1, 'weight_decay': 0.0001, 'clip_grad': 1.0, 'use_fp16': True, 'early_stop_patience': 8, 'min_epochs_before_stop': 10, 'warmup_steps': 2000, 'steps_per_epoch': 14844, 'optimizer_base_lr': 1.0, 'seed': 42}
Checkpoint helpers ready.


In [ ]:
# ── MAIN TRAINING LOOP — ONE CLEAN LINE PER EPOCH ─────────────────────

import contextlib
import io
import math
import time
import pandas as pd


SHOW_EVAL_EXAMPLES = False

# Start using chrF++ for checkpointing from this epoch.
CHRF_TRACK_START = max(1, MIN_EPOCHS_BEFORE_STOP // 2)

# Prevent duplicate rows when rerunning after resuming.
training_log = [
    row for row in training_log
    if int(row.get("epoch", -1)) < START_EPOCH
]

last_completed_epoch = START_EPOCH - 1

print(f"\nTraining: {RUN_NAME}")
print(f"Epochs: {START_EPOCH}-{MAX_EPOCHS} | Device: {device}\n")


for epoch in range(START_EPOCH, MAX_EPOCHS + 1):
    epoch_start = time.time()

    # ── 1. Train ───────────────────────────────────────────────────────
    # Hide batch loaders, step counts and internal training output.
    with (
        contextlib.redirect_stdout(io.StringIO()),
        contextlib.redirect_stderr(io.StringIO()),
    ):
        tr_loss, GLOBAL_STEP = train_epoch(
            model,
            train_loader,
            optimizer,
            scheduler,
            scaler,
            criterion,
            GLOBAL_STEP,
            GRADIENT_ACCUM,
        )

    if not math.isfinite(tr_loss):
        raise RuntimeError(
            f"Training loss became invalid at epoch {epoch}: {tr_loss}"
        )

    # ── 2. Validate ────────────────────────────────────────────────────
    # Hide validation loader and internal output.
    with (
        contextlib.redirect_stdout(io.StringIO()),
        contextlib.redirect_stderr(io.StringIO()),
    ):
        vl_loss = validate(
            model,
            val_loader,
            criterion,
        )

    if not math.isfinite(vl_loss):
        raise RuntimeError(
            f"Validation loss became invalid at epoch {epoch}: {vl_loss}"
        )

    # ── 3. Quick FLORES evaluation ─────────────────────────────────────
    if SHOW_EVAL_EXAMPLES:
        bleu, chrf, empty_pct, avg_len, unique_ratio = (
            quick_flores_eval(model)
        )
    else:
        # Hide evaluation loader and SRC / REF / HYP examples.
        with (
            contextlib.redirect_stdout(io.StringIO()),
            contextlib.redirect_stderr(io.StringIO()),
        ):
            bleu, chrf, empty_pct, avg_len, unique_ratio = (
                quick_flores_eval(model)
            )

    elapsed = time.time() - epoch_start
    current_lr = scheduler.get_last_lr()[0]

    metrics = [
        bleu,
        chrf,
        empty_pct,
        avg_len,
        unique_ratio,
    ]

    eval_valid = all(
        math.isfinite(float(value))
        for value in metrics
    )

    # ── 4. Check improvements ──────────────────────────────────────────
    val_improved = vl_loss < BEST_VAL_LOSS

    chrf_tracking_active = (
        eval_valid
        and epoch >= CHRF_TRACK_START
    )

    chrf_improved = (
        chrf_tracking_active
        and chrf > BEST_DEV_CHRF
    )

    if val_improved:
        BEST_VAL_LOSS = vl_loss

    if chrf_tracking_active:
        if chrf_improved:
            BEST_DEV_CHRF = chrf
            NO_IMPROVE = 0
        else:
            NO_IMPROVE += 1

    # ── 5. Save training log ───────────────────────────────────────────
    training_log.append({
        "epoch": epoch,
        "global_step": GLOBAL_STEP,
        "lr": current_lr,
        "train_loss": tr_loss,
        "val_loss": vl_loss,
        "dev_bleu": bleu,
        "dev_chrf_pp": chrf,
        "empty_pct": empty_pct,
        "avg_len": avg_len,
        "unique_ratio": unique_ratio,
        "best_val_loss": BEST_VAL_LOSS,
        "best_dev_chrf": BEST_DEV_CHRF,
        "no_improve": NO_IMPROVE,
        "elapsed_s": elapsed,
    })

    # ── 6. Save best validation checkpoint ─────────────────────────────
    if val_improved:
        save_checkpoint(
            CKPT_BEST_VAL,
            epoch,
            GLOBAL_STEP,
            model,
            optimizer,
            scheduler,
            scaler,
            BEST_VAL_LOSS,
            BEST_DEV_CHRF,
            NO_IMPROVE,
        )

    # ── 7. Save best chrF++ checkpoint ─────────────────────────────────
    if chrf_improved:
        save_checkpoint(
            CKPT_BEST_CHRF,
            epoch,
            GLOBAL_STEP,
            model,
            optimizer,
            scheduler,
            scaler,
            BEST_VAL_LOSS,
            BEST_DEV_CHRF,
            NO_IMPROVE,
        )

    # ── 8. Always save latest checkpoint ───────────────────────────────
    save_checkpoint(
        CKPT_LATEST,
        epoch,
        GLOBAL_STEP,
        model,
        optimizer,
        scheduler,
        scaler,
        BEST_VAL_LOSS,
        BEST_DEV_CHRF,
        NO_IMPROVE,
    )

    # ── 9. Save CSV ────────────────────────────────────────────────────
    pd.DataFrame(training_log).to_csv(
        str(LOG_CSV),
        index=False,
    )

    last_completed_epoch = epoch

    # ── 10. Only required epoch output ─────────────────────────────────
    saved_status = ""

    if val_improved and chrf_improved:
        saved_status = " | saved best-val + best-chrF"
    elif val_improved:
        saved_status = " | saved best-val"
    elif chrf_improved:
        saved_status = " | saved best-chrF"

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"train {tr_loss:.4f} | "
        f"val {vl_loss:.4f} | "
        f"chrF++ {chrf:.2f} | "
        f"BLEU {bleu:.2f} | "
        f"{elapsed:.0f}s"
        f"{saved_status}"
    )

    # ── 11. Early stopping ─────────────────────────────────────────────
    should_stop = (
        chrf_tracking_active
        and epoch >= MIN_EPOCHS_BEFORE_STOP
        and NO_IMPROVE >= EARLY_STOP_PATIENCE
    )

    if should_stop:
        print(
            f"\nEarly stopping at epoch {epoch}: "
            f"chrF++ did not improve for "
            f"{NO_IMPROVE} eligible epochs."
        )
        break


# ── FINAL SAVE ─────────────────────────────────────────────────────────

pd.DataFrame(training_log).to_csv(
    str(LOG_CSV),
    index=False,
)

print("\nTraining complete")
print(f"Best validation loss : {BEST_VAL_LOSS:.4f}")
print(f"Best chrF++          : {BEST_DEV_CHRF:.2f}")   
print(f"Final epoch          : {last_completed_epoch}")
print(f"Latest checkpoint    : {CKPT_LATEST.name}")
print(f"Training log         : {LOG_CSV.name}")


Training: student_beam_M10_merged_optA_fixed_v2
Epochs: 2-60 | Device: cuda



Train:   0%|          | 0/29688 [00:00<?, ?it/s]

Val:   0%|          | 0/1563 [00:00<?, ?it/s]

QuickEval:   0%|          | 0/200 [00:00<?, ?it/s]

  Saved: student_beam_M10_merged_optA_fixed_v2_best_val.pt
  Saved: student_beam_M10_merged_optA_fixed_v2_latest.pt
Epoch 02/60 | train 3.3543 | val 2.9161 | chrF++ 51.49 | BLEU 26.94 | 3554s | saved best-val


Train:   0%|          | 0/29688 [00:00<?, ?it/s]

In [ ]:
from IPython.display import FileLink, display

# Link for the best validation model
display(FileLink(r'models/student_run/student_beam_M10_merged_optA_fixed_v2/student_beam_M10_merged_optA_fixed_v2_best_val.pt'))

# Link for the latest model
display(FileLink(r'models/student_run/student_beam_M10_merged_optA_fixed_v2/student_beam_M10_merged_optA_fixed_v2_latest.pt'))

In [ ]:
# ── OUTPUT FILE MANIFEST ────────────────────────────────────────────────
print('=== Output File Manifest ===')
print()
_all_files = []
for _d in [CKPT_DIR, RESULTS_DIR]:
    for _f in sorted(_d.rglob('*')):
        if _f.is_file():
            _all_files.append(_f)
            print(' {:60s}  {:>8.1f} KB'.format(str(_f), _f.stat().st_size / 1024))
print()
print('Total files:', len(_all_files))
print()
print('=== Kaggle Output Dataset Instructions ===')
print('1. After this run completes, go to the Kaggle notebook output tab.')
print('2. Find the checkpoint files under /kaggle/working/models/{}/'.format(RUN_NAME))
print('3. Click "New Dataset" and add all .pt files and the training_log CSV.')
print('4. Name the dataset to match your project (e.g., mhd-models-v2).')
print('5. Attach this dataset to the next notebook (03c_evaluate) as an input.')
print()
print('Best checkpoint (val loss) :', CKPT_BEST_VAL)
print('Best checkpoint (chrF++)  :', CKPT_BEST_CHRF)
print('Latest checkpoint         :', CKPT_LATEST)
print('Training log CSV          :', LOG_CSV)